In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

PERSIST_DIR = r"C:\Users\sanja\Desktop\LEX-PROJECT\chroma_db"
COLLECTION_NAME = "indian_laws"   # the one you used during rebuild

embedding = OpenAIEmbeddings()

vectordb = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embedding,
    collection_name=COLLECTION_NAME
)

print("Vector count:", vectordb._collection.count())


C:\Users\sanja\Desktop\LEX-PROJECT\venv\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 0.4. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  warn_deprecated(


Vector count: 3159


In [7]:
def infer_law(metadata):
    src = metadata.get("source", "").lower()
    if "constitution" in src:
        return "Constitution"
    if "bnss" in src:
        return "BNSS"
    if "bns" in src:
        return "BNS"
    return "UNKNOWN"


In [8]:
docs = retrieve("punishment for murder", k=5)

for d in docs:
    print(infer_law(d.metadata), d.metadata["page"])


BNS 33
BNS 7
BNS 33
BNS 33
BNS 33


In [11]:
def retrieve_with_law(query, k=5):
    docs = vectordb.similarity_search(query, k=k)

    results = []
    for d in docs:
        results.append({
            "law": infer_law(d.metadata),
            "page": d.metadata.get("page"),
            "content": d.page_content
        })

    return results


In [12]:
results = retrieve_with_law("punishment for murder")

for r in results:
    print(r["law"], "page:", r["page"])


BNS page: 33
BNS page: 7
BNS page: 33
BNS page: 33
BNS page: 33


In [41]:
chat_history = []              # stores ALL chats
conversation_summary = ""      # compressed long-term memory
last_topic = ""                # tracks last legal concept


In [42]:
def extract_topic(text):
    topics = [
        "murder",
        "punishment",
        "intention",
        "minor",
        "woman",
        "constitutional",
        "fundamental rights"
    ]
    for t in topics:
        if t in text.lower():
            return t
    return ""


In [43]:
def resolve_followup(query):
    vague_phrases = [
        "explain that",
        "explain it",
        "that simply",
        "explain simply",
        "explain again"
    ]

    if any(v in query.lower() for v in vague_phrases) and last_topic:
        return f"Explain {last_topic} in simple words"
    
    return query


In [44]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def update_summary(summary, user_msg, assistant_msg):
    prompt = f"""
You are summarizing a legal conversation.

Current summary:
{summary}

New exchange:
User: {user_msg}
Assistant: {assistant_msg}

Update the summary concisely, preserving legal meaning.
"""
    return llm.invoke(prompt).content


In [45]:
def answer_legal_query(query):
    global conversation_summary, last_topic

    # ✅ Resolve vague follow-ups FIRST
    query = resolve_followup(query)

    # ✅ Retrieve legal context
    retrieved = retrieve_with_law(query)

    context = ""
    for r in retrieved:
        context += f"""
LAW: {r['law']}
PAGE: {r['page']}

{r['content']}
---
"""

    # ✅ Recent chat turns (last 5)
    recent = ""
    for turn in chat_history[-5:]:
        recent += f"""
User: {turn['user']}
Assistant: {turn['assistant']}
"""

    # ✅ Final prompt
    prompt = f"""
You are a Legal Assistant for Indian Law.

Conversation summary:
{conversation_summary}

Recent conversation:
{recent}

Answer ONLY using the legal context below.
Stay on topic. Do not introduce unrelated laws.

Legal context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt).content

    # ✅ Store full history
    chat_history.append({
        "user": query,
        "assistant": response
    })

    # ✅ Update topic
    topic = extract_topic(query)
    if topic:
        last_topic = topic

    # ✅ Update long-term summary
    conversation_summary = update_summary(
        conversation_summary, query, response
    )

    return response
